# Student Dropout Prediction: Multi-Model Classification Analysis

**Dataset:** Higher education student records (4,424 students, 36 features)  
**Objective:** Predict student outcome (Dropout, Enrolled, Graduate) from demographic, academic, and financial data  
**Models evaluated:** Logistic Regression, Support Vector Machine, Random Forest  
**Author:** Rijan Paudel

---

## Business Context

Student dropout is a significant institutional challenge. Early identification of at-risk students allows universities to intervene with targeted support, reducing attrition and improving graduation outcomes. This analysis builds a classification model that flags dropout risk using data available at enrollment and after the first academic semester.

**Key questions this analysis answers:**
1. Which student attributes most strongly predict dropout?
2. How accurately can outcome be predicted from available data?
3. What actionable patterns exist in the data for institutional policy?

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# Consistent plot styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

In [ ]:
df = pd.read_csv("student.csv")

print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nTarget classes: {df['Target'].unique().tolist()}")
print(f"\nTarget distribution:")
print(df['Target'].value_counts())
print(f"\nOverall dropout rate: {(df['Target'] == 'Dropout').mean():.1%}")

## 2. Exploratory Data Analysis

Before building any model, understanding the data distribution is essential. Three areas are worth examining:
- The target variable balance (are classes skewed?)
- The relationship between financial indicators and dropout
- Academic performance patterns across outcome groups

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "None - dataset is complete")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Target distribution
df['Target'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c', '#3498db', '#2ecc71'])
axes[0].set_title("Student Outcome Distribution")
axes[0].set_xlabel("")
axes[0].tick_params(axis='x', rotation=0)

# Scholarship vs dropout
scholarship_dropout = df.groupby('Scholarship holder')['Target'].value_counts(normalize=True).unstack()
scholarship_dropout.plot(kind='bar', ax=axes[1], colormap='Set2')
axes[1].set_title("Outcome by Scholarship Status\n(0=No, 1=Yes)")
axes[1].set_xlabel("")
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(loc='upper right', fontsize=8)

# Tuition status vs dropout
tuition_dropout = df.groupby('Tuition fees up to date')['Target'].value_counts(normalize=True).unstack()
tuition_dropout.plot(kind='bar', ax=axes[2], colormap='Set1')
axes[2].set_title("Outcome by Tuition Status\n(0=Overdue, 1=Up to date)")
axes[2].set_xlabel("")
axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

**Key observations from EDA:**

- The dataset is moderately imbalanced: 50% Graduates, 32% Dropouts, 18% Enrolled. This imbalance means accuracy alone is not a sufficient evaluation metric - precision and recall per class matter.
- **Scholarship status is a strong protective factor.** Students with scholarships have a 12.2% dropout rate, compared to 38.7% for non-scholarship students.
- **Tuition payment status is the single strongest binary signal.** Students with overdue tuition have an 86.6% dropout rate. Students current on tuition drop out at just 24.7%. This single field almost cleanly separates dropout from non-dropout outcomes.
- **Debt is a high-risk indicator.** Students flagged as debtors drop out at a 62% rate, versus 28.3% for non-debtors.

In [ ]:
# Age distribution by outcome
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(x="Target", y="Age at enrollment", data=df, ax=axes[0], palette="Set2")
axes[0].set_title("Enrollment Age by Student Outcome")
axes[0].set_xlabel("Outcome")

sns.boxplot(x="Target", y="Curricular units 1st sem (grade)", data=df, ax=axes[1], palette="Set2")
axes[1].set_title("1st Semester Grade by Student Outcome")
axes[1].set_xlabel("Outcome")

plt.tight_layout()
plt.show()

**Age and academic performance patterns:**

- Dropout students enroll at a median age of 23, compared to 19 for graduates. Older students at enrollment face higher dropout risk, likely due to competing life demands (employment, family obligations).
- First-semester grades show a clear separation: graduates score substantially higher than dropout students. This confirms that early academic performance is a leading indicator of final outcome - interventions in the first semester have the highest leverage.

## 3. Data Preprocessing

The dataset contains categorical columns that require encoding before modelling. StandardScaler is applied so that distance-based models (SVM, Logistic Regression) are not distorted by features with large numeric ranges (e.g., admission grade vs. binary flags).

In [ ]:
X = df.drop("Target", axis=1)
y = df["Target"]

# Encode target labels to integers
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y)
print("Target encoding:", dict(zip(target_encoder.classes_, range(len(target_encoder.classes_)))))

# Encode any remaining object-type columns
label_encoders = {}
for col in X.columns:
    if X[col].dtype == 'object':
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])
        label_encoders[col] = le

print(f"\nFeatures after encoding: {X.shape[1]}")
print("All features are now numeric.")

In [ ]:
# 80/20 train-test split, stratified to preserve class ratios
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print("Stratified split preserves class distribution across train and test.")

## 4. Model Training and Comparison

Three models are trained and compared:
- **Logistic Regression** - linear baseline, interpretable, computationally cheap
- **Support Vector Machine (RBF kernel)** - non-linear boundary, typically strong on tabular data
- **Random Forest** - ensemble of decision trees, robust to outliers, produces feature importance scores

All three are evaluated on the held-out test set.

In [ ]:
# Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)
lr_acc = accuracy_score(y_test, y_pred_lr)
print(f"Logistic Regression Accuracy: {lr_acc:.4f}")

In [ ]:
# SVM
svm_model = SVC(kernel='rbf', random_state=42)
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)
svm_acc = accuracy_score(y_test, y_pred_svm)
print(f"SVM Accuracy: {svm_acc:.4f}")

In [ ]:
# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest Accuracy: {rf_acc:.4f}")

In [ ]:
# Model comparison summary
results = pd.DataFrame({
    "Model": ["Logistic Regression", "SVM (RBF)", "Random Forest"],
    "Accuracy": [lr_acc, svm_acc, rf_acc]
}).sort_values("Accuracy", ascending=False)

print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#2ecc71' if v == results['Accuracy'].max() else '#95a5a6' for v in results['Accuracy']]
ax.barh(results['Model'], results['Accuracy'], color=colors)
ax.set_xlim(0.5, 0.85)
ax.set_xlabel("Test Accuracy")
ax.set_title("Model Accuracy Comparison")
for i, v in enumerate(results['Accuracy']):
    ax.text(v + 0.002, i, f"{v:.2%}", va='center')
plt.tight_layout()
plt.show()

**Model comparison findings:**

All three models perform within a close range (75-77%), which suggests the task has an inherent ceiling at this feature set - the 'Enrolled' class is genuinely difficult to separate from the other two, as enrollment is an intermediate state.

Logistic Regression performs marginally best (76.8%), followed by Random Forest (76.4%) and SVM (75.8%). Given that Logistic Regression and Random Forest are within 0.5% of each other, **Random Forest is the preferred model for deployment** because it is:
- More robust to outliers (important given the age outliers seen in EDA)
- Non-parametric, making no assumptions about feature distributions
- Capable of producing feature importance scores, which are essential for institutional interpretation

## 5. Detailed Model Evaluation

Accuracy alone is misleading on an imbalanced dataset. The confusion matrix and per-class metrics reveal where each model struggles.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, preds, title, cmap in zip(
    axes,
    [y_pred_lr, y_pred_svm, y_pred_rf],
    ["Logistic Regression", "SVM", "Random Forest"],
    ["Blues", "Greens", "Oranges"]
):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, ax=ax,
                xticklabels=target_encoder.classes_,
                yticklabels=target_encoder.classes_)
    ax.set_title(f"Confusion Matrix\n{title}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()

In [ ]:
print("Random Forest - Full Classification Report")
print("=" * 50)
print(classification_report(y_test, y_pred_rf, target_names=target_encoder.classes_))

**Evaluation findings:**

- **Dropout detection (recall: 75%)** - The model correctly identifies 75% of actual dropout students. For an early-warning system, this is a reasonable baseline; the 25% it misses would need further feature engineering to capture.
- **Graduate prediction (recall: 92%)** - The model is highly reliable at identifying students who will graduate, making it useful for positive outcome tracking.
- **Enrolled class (recall: 35%)** - The 'Enrolled' class is the weakest prediction, which is expected: a student currently enrolled has not yet resolved their trajectory. This class would benefit from time-series features (semester-by-semester progression data) not available in this static snapshot.

## 6. Feature Importance Analysis

Understanding *which features drive predictions* is as important as the accuracy number. Feature importance from Random Forest quantifies each variable's contribution to splitting decisions across all 100 trees.

In [ ]:
importances = rf_model.feature_importances_
feat_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importances
}).sort_values("Importance", ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(x="Importance", y="Feature", data=feat_df.head(15), palette="coolwarm_r", ax=ax)
ax.set_title("Top 15 Features - Random Forest Importance Scores")
ax.set_xlabel("Mean Decrease in Impurity")
plt.tight_layout()
plt.show()

print("\nTop 10 features with importance scores:")
print(feat_df.head(10).to_string(index=False))

**Feature importance findings:**

The top predictors are overwhelmingly academic performance metrics from both semesters, with financial indicators following:

| Rank | Feature | Importance | Interpretation |
|------|---------|-----------|----------------|
| 1 | 2nd sem units approved | 14.2% | Core academic signal - failing to pass units is the strongest predictor |
| 2 | 2nd sem grade | 10.9% | Overall academic trajectory by end of first year |
| 3 | 1st sem units approved | 9.2% | Early warning signal available within the first 4 months |
| 4 | 1st sem grade | 6.0% | Confirms 1st semester is a reliable early indicator |
| 5 | Admission grade | 4.4% | Entry-level academic preparedness has lasting impact |
| 6 | Age at enrollment | 4.0% | Older students face higher dropout risk |
| 7 | Tuition fees up to date | 3.9% | Financial stability signal |

**The practical implication:** A simplified early-warning model using only 1st semester academic data (available at ~4 months) could provide timely intervention triggers without waiting for full-year data.

## 7. Feature Correlation Structure

In [ ]:
df_encoded = df.copy()
for col in df_encoded.columns:
    if df_encoded[col].dtype == 'object':
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col])

# Focus on top correlated features with target
target_corr = df_encoded.corr()['Target'].abs().sort_values(ascending=False)
top_features = target_corr.head(12).index.tolist()

plt.figure(figsize=(10, 8))
sns.heatmap(
    df_encoded[top_features].corr(),
    cmap="coolwarm", annot=True, fmt=".2f",
    linewidths=0.5, annot_kws={"size": 8}
)
plt.title("Correlation Heatmap - Top 12 Features by Target Correlation")
plt.tight_layout()
plt.show()

## 8. Summary and Business Recommendations

### Model Performance
Three classification models were evaluated on a 4,424-student dataset. Logistic Regression achieved the highest accuracy (76.8%), with Random Forest preferred for deployment due to its robustness and interpretability through feature importance scoring. The 'Enrolled' class is inherently difficult to classify from static enrollment-time features.

### Key Findings

**Finding 1: Academic performance in the first semester is the highest-leverage early warning signal.**  
The top 4 features by importance are all academic metrics from semesters 1 and 2. Institutions should flag students who complete fewer than the expected number of curricular units by the end of semester 1 - this signal is available just 4 months into enrollment.

**Finding 2: Financial indicators have outsized dropout impact.**  
Students with overdue tuition drop out at an 86.6% rate. Students with outstanding debt drop out at 62%. These are the two most actionable intervention targets: a financial aid outreach triggered by early payment delays could materially reduce dropout rates.

**Finding 3: Scholarships are protective at scale.**  
Scholarship holders drop out at 12.2% versus 38.7% for non-recipients. Expanding scholarship coverage to borderline-eligible students in high-risk financial categories could reduce institutional dropout rates significantly.

**Finding 4: Age at enrollment is a demographic risk factor.**  
Dropout students enroll at a median age of 23 versus 19 for graduates. Mature-age students likely face employment or family constraints not captured in this dataset. Targeted support programs (flexible scheduling, part-time pathways) for students enrolling over age 25 would address an identifiable risk cohort.

### Limitations and Next Steps
- The 'Enrolled' class could be improved with time-series data tracking semester-by-semester progression
- Hyperparameter tuning (GridSearchCV on RF `max_depth`, `min_samples_split`) may recover 1-2% accuracy
- SMOTE or class-weight adjustments could improve minority class recall
- Feature engineering: a combined 'academic risk score' from sem 1 data could improve early detection